# 🔧 Setup

In [53]:
from pipeline import get_preprocessing_pipeline, get_feature_names, get_target_name, build_model_pipeline

from sklearn.model_selection import (train_test_split, GridSearchCV)
from sklearn.base import BaseEstimator
from sklearn.metrics import (accuracy_score, recall_score, precision_score, average_precision_score, precision_recall_curve, auc)

from sklearn.linear_model import (LogisticRegression)


import data_loader

PATH_DATA ='data/raw/Telco_customer_churn.xlsx'
SEED = 4586

# 📦 Train-Test split

In [54]:
df = data_loader.load_churn_data(PATH_DATA)

In [55]:
X = df[get_feature_names(False)]
y = df[get_target_name(False)]

In [56]:
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=SEED, stratify=y)
y_test.value_counts()

Churn Value
0    1294
1     467
Name: count, dtype: int64

# 📝 Baseline

In [57]:
def train_test_model(model, X_train, y_train, X_test, y_test):
    model.fit(X_train, y_train)
    
    y_pred = model.predict(X_test)

    acc = accuracy_score(y_pred, y_test)
    prec = precision_score(y_pred, y_test)
    rec = recall_score(y_pred, y_test)
    print(acc, prec, rec)

In [58]:
param_grids = {
    'lr':{
        'model__solver': ['saga'],
        'model__l1_ratio': [0, 0.25, 0.5, 0.75, 1],  # 0 = pure L2, 1 = pure L1
        'model__C': [0.1, 1.0, 10.0],
        'model__max_iter': [1000],  # Increased so saga converges cleanly
        'model__class_weight':['balanced']
        }
    }

In [59]:
model_lr = LogisticRegression()
pipeline_lr = build_model_pipeline(
    model_lr,
    scale=True
    )

train_test_model(pipeline_lr, X_train, y_train, X_test, y_test)

0.8097671777399205 0.556745182012848 0.6701030927835051


In [ ]:
grid_search = GridSearchCV(
    estimator = pipeline_lr,
    param_grid = param_grids['lr'],
    cv=5,
    scoring='average_precision',
    n_jobs=-1
    )

grid_search.fit(X_train, y_train)

print("Best Parameters:", grid_search.best_params_)
print("Best Cross-Val AUC-PR:", round(grid_search.best_score_, 4))

Best Parameters: {'model__C': 0.1, 'model__class_weight': 'balanced', 'model__l1_ratio': 0.5, 'model__max_iter': 1000, 'model__solver': 'saga'}
Best Cross-Val Recall: 0.6708
